# MCD-rPPG Dataset - Exploratory Data Analysis (EDA)

## Comprehensive Dataset Showcase

This notebook provides a thorough exploration of the **MCD-rPPG (Multi-Camera Dataset for Remote Photoplethysmography)** dataset.

---

### Setup and Configuration

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import cv2
import skvideo.io
from IPython.display import Video, display, HTML
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Configuration
DB_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/db.csv'
VIDEOS_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/video/'
PPG_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/ppg/'
PPG_SYNC_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/ppg_sync/'
ECG_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/ecg/'
METADATA_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/metadata/'

print('Setup complete!')
print(f'DB Path: {DB_PATH}')
print(f'Videos Path: {VIDEOS_PATH}')

## 1. File Structure Analysis

In [ ]:
def analyze_directory(path, name, max_depth=2):
    if not os.path.exists(path):
        print(f'Directory does not exist: {path}')
        return
    
    total_files = 0
    total_size = 0
    
    for root, dirs, files in os.walk(path):
        level = root.replace(path, '').count(os.sep)
        if level > max_depth:
            continue
        
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        
        for file in files[:5]:
            file_path = os.path.join(root, file)
            file_size = os.path.getsize(file_path)
            total_files += 1
            total_size += file_size
            print(f'{indent}  {file} ({file_size / 1024 / 1024:.2f} MB)')
        
        if len(files) > 5:
            print(f'{indent}  ... and {len(files) - 5} more files')
            total_files += len(files) - 5
    
    print(f'Total: {total_files} files, {total_size / 1024 / 1024 / 1024:.2f} GB')

print('VIDEOS:')
analyze_directory(VIDEOS_PATH, 'VIDEOS')

print('PPG SIGNALS:')
analyze_directory(PPG_PATH, 'PPG')

print('ECG SIGNALS:')
analyze_directory(ECG_PATH, 'ECG')

print('METADATA:')
analyze_directory(METADATA_PATH, 'METADATA')

## 2. Database Analysis

In [ ]:
if os.path.exists(DB_PATH):
    df_db = pd.read_csv(DB_PATH)
    print(f'Database loaded: {df_db.shape}')
    print(f'Columns: {list(df_db.columns)}')
    display(df_db.head())
    display(df_db.describe())
    
    # Missing values
    missing = df_db.isnull().sum()
    missing_pct = (missing / len(df_db)) * 100
    missing_df = pd.DataFrame({'Missing': missing, 'Percentage': missing_pct})
    display(missing_df[missing_df['Missing'] > 0].sort_values('Missing', ascending=False))
else:
    print(f'Database not found at: {DB_PATH}')

## 3. Subject Demographics

In [ ]:
if os.path.exists(DB_PATH):
    df_db['subject_id'] = df_db['file'].apply(lambda x: os.path.basename(x).split('__')[1] if '__' in x else 'unknown')
    df_db['condition'] = df_db['file'].apply(lambda x: 'after' if 'after' in x else 'before' if 'before' in x else 'unknown')
    
    print(f'Total unique subjects: {df_db["subject_id"].nunique()}')
    print(f'Total recordings: {len(df_db)}')
    print(f'Recordings per subject: {len(df_db) / df_db["subject_id"].nunique():.1f}')
    
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    df_db['condition'].value_counts().plot(kind='bar', color=['skyblue', 'salmon'])
    plt.title('Condition Distribution')
    
    plt.subplot(1, 2, 2)
    df_db['subject_id'].value_counts().value_counts().sort_index().plot(kind='bar', color='lightgreen')
    plt.title('Recordings per Subject')
    plt.tight_layout()
    plt.show()

## 4. Video Data Analysis

In [ ]:
if os.path.exists(DB_PATH):
    print('Video Statistics:')
    video_stats = df_db[['video_dtype', 'video_shape', 'video_min', 'video_max', 'video_mean', 'video_std', 'video_duration']]
    display(video_stats.describe())
    
    plt.figure(figsize=(15, 10))
    plt.subplot(2, 3, 1)
    plt.hist(df_db['video_duration'], bins=50, color='skyblue', edgecolor='black')
    plt.title('Video Duration Distribution')
    
    plt.subplot(2, 3, 2)
    plt.hist(df_db['video_mean'], bins=50, color='salmon', edgecolor='black')
    plt.title('Video Mean Intensity')
    
    plt.subplot(2, 3, 3)
    plt.hist(df_db['video_std'], bins=50, color='lightgreen', edgecolor='black')
    plt.title('Video Std Dev')
    
    plt.subplot(2, 3, 4)
    plt.scatter(df_db['video_mean'], df_db['video_std'], alpha=0.5, c=df_db['video_duration'], cmap='viridis')
    plt.colorbar(label='Duration')
    plt.title('Mean vs Std Dev')
    
    plt.subplot(2, 3, 5)
    plt.scatter(df_db['patient_id'], df_db['video_duration'], alpha=0.5, color='purple')
    plt.title('Duration by Patient ID')
    plt.xticks(rotation=45)
    
    if 'fold' in df_db.columns:
        plt.subplot(2, 3, 6)
        sns.boxplot(x='fold', y='video_duration', data=df_db, palette='Set2')
        plt.title('Duration by Fold')
    
    plt.tight_layout()
    plt.show()

## 5. PPG Signal Analysis

In [ ]:
if os.path.exists(DB_PATH):
    print('PPG Statistics:')
    ppg_stats = df_db[['ppg_dtype', 'ppg_shape', 'ppg_min', 'ppg_max', 'ppg_mean', 'ppg_std']]
    display(ppg_stats.describe())
    
    df_db['ppg_range'] = df_db['ppg_max'] - df_db['ppg_min']
    
    plt.figure(figsize=(15, 10))
    plt.subplot(2, 3, 1)
    plt.hist(df_db['ppg_mean'], bins=50, color='red', edgecolor='black', alpha=0.7)
    plt.title('PPG Mean Distribution')
    
    plt.subplot(2, 3, 2)
    plt.hist(df_db['ppg_std'], bins=50, color='pink', edgecolor='black', alpha=0.7)
    plt.title('PPG Std Dev Distribution')
    
    plt.subplot(2, 3, 3)
    plt.hist(df_db['ppg_min'], bins=50, color='darkred', edgecolor='black', alpha=0.7)
    plt.title('PPG Minimum Distribution')
    
    plt.subplot(2, 3, 4)
    plt.hist(df_db['ppg_max'], bins=50, color='coral', edgecolor='black', alpha=0.7)
    plt.title('PPG Maximum Distribution')
    
    plt.subplot(2, 3, 5)
    plt.scatter(df_db['ppg_mean'], df_db['ppg_std'], alpha=0.5, c=df_db['video_duration'], cmap='Reds')
    plt.colorbar(label='Video Duration')
    plt.title('PPG Mean vs Std Dev')
    
    plt.subplot(2, 3, 6)
    plt.hist(df_db['ppg_range'], bins=50, color='brown', edgecolor='black', alpha=0.7)
    plt.title('PPG Range Distribution')
    
    plt.tight_layout()
    plt.show()

## 6. ECG Signal Analysis

In [ ]:
if os.path.exists(ECG_PATH):
    ecg_files = []
    for root, dirs, files in os.walk(ECG_PATH):
        for file in files[:10]:
            ecg_files.append(os.path.join(root, file))
    
    print(f'Found {len(ecg_files)} ECG files')
    for file in ecg_files[:5]:
        size = os.path.getsize(file) / 1024 / 1024
        print(f'  {file} ({size:.2f} MB)')
    
    if ecg_files:
        try:
            sample_ecg = np.load(ecg_files[0])
            print(f'Sample ECG: shape={sample_ecg.shape}, dtype={sample_ecg.dtype}')
            print(f'Min={sample_ecg.min():.2f}, Max={sample_ecg.max():.2f}, Mean={sample_ecg.mean():.2f}')
            
            plt.figure(figsize=(15, 4))
            plt.plot(sample_ecg[:1000], color='green', linewidth=1)
            plt.title('ECG Signal (First 1000 samples)')
            plt.xlabel('Sample Index')
            plt.ylabel('ECG Value')
            plt.grid(True, alpha=0.3)
            plt.show()
            
            time = np.arange(len(sample_ecg)) / 100.0
            plt.figure(figsize=(15, 4))
            plt.plot(time[:10], sample_ecg[:1000], color='green', linewidth=1)
            plt.title('ECG Signal (First 10 seconds)')
            plt.xlabel('Time (seconds)')
            plt.ylabel('ECG Value')
            plt.grid(True, alpha=0.3)
            plt.show()
        except Exception as e:
            print(f'Error loading ECG: {e}')

## 7. Landmarks Analysis

In [ ]:
if os.path.exists(DB_PATH):
    print('Landmarks Statistics:')
    landmarks_stats = df_db[['landmarks_dtype', 'landmarks_shape', 'landmarks_min', 'landmarks_max', 'landmarks_mean', 'landmarks_std', 'face_square']]
    display(landmarks_stats.describe())
    
    plt.figure(figsize=(15, 10))
    plt.subplot(2, 3, 1)
    plt.hist(df_db['landmarks_mean'], bins=50, color='purple', edgecolor='black', alpha=0.7)
    plt.title('Landmarks Mean')
    
    plt.subplot(2, 3, 2)
    plt.hist(df_db['landmarks_std'], bins=50, color='violet', edgecolor='black', alpha=0.7)
    plt.title('Landmarks Std Dev')
    
    plt.subplot(2, 3, 3)
    plt.hist(df_db['face_square'], bins=50, color='indigo', edgecolor='black', alpha=0.7)
    plt.title('Face Square Distribution')
    
    plt.subplot(2, 3, 4)
    plt.scatter(df_db['landmarks_mean'], df_db['face_square'], alpha=0.5, c=df_db['video_duration'], cmap='viridis')
    plt.colorbar(label='Video Duration')
    plt.title('Landmarks Mean vs Face Square')
    
    plt.subplot(2, 3, 5)
    plt.scatter(df_db['landmarks_std'], df_db['face_square'], alpha=0.5, c=df_db['ppg_mean'], cmap='Reds')
    plt.colorbar(label='PPG Mean')
    plt.title('Landmarks Std vs Face Square')
    
    plt.subplot(2, 3, 6)
    plt.scatter(df_db['face_square'], df_db['video_duration'], alpha=0.5, color='blue')
    plt.title('Face Square vs Video Duration')
    
    plt.tight_layout()
    plt.show()

## 8. Health Biomarkers Analysis

In [ ]:
metadata_files = []
if os.path.exists(METADATA_PATH):
    for root, dirs, files in os.walk(METADATA_PATH):
        for file in files:
            metadata_files.append(os.path.join(root, file))
    
    print(f'Metadata files found: {len(metadata_files)}')
    for file in metadata_files[:10]:
        print(f'  {file}')

biomarkers_df = None
subjects_df = None

for filename in ['biomarkers.csv', 'subjects.csv', 'metadata.csv', 'vitals.csv']:
    test_path = os.path.join(METADATA_PATH, filename)
    if os.path.exists(test_path):
        try:
            df = pd.read_csv(test_path)
            print(f'Loaded {filename}: shape={df.shape}')
            display(df.head())
            
            if 'biomarker' in filename.lower():
                biomarkers_df = df
            elif 'subject' in filename.lower():
                subjects_df = df
        except Exception as e:
            print(f'Error loading {filename}: {e}')

print('Expected Health Biomarkers:')
expected_biomarkers = {
    'Cardiovascular': ['Systolic Blood Pressure', 'Diastolic Blood Pressure', 'Heart Rate', 'Arterial Stiffness'],
    'Respiratory': ['Oxygen Saturation (SpO2)', 'Respiratory Rate'],
    'Metabolic': ['Glucose', 'Glycated Hemoglobin (HbA1c)', 'Cholesterol'],
    'Physiological': ['Temperature'],
    'Psychological': ['Stress Level (PSM-25)'],
    'Demographic': ['Age', 'Sex', 'BMI']
}

for category, biomarkers in expected_biomarkers.items():
    print(f'  {category}:')
    for biomarker in biomarkers:
        print(f'    - {biomarker}')

## 9. Synchronization Analysis

In [ ]:
sync_files = []
if os.path.exists(METADATA_PATH):
    for root, dirs, files in os.walk(METADATA_PATH):
        for file in files:
            if 'sync' in file.lower():
                sync_files.append(os.path.join(root, file))
    
    print(f'Synchronization files: {len(sync_files)}')
    for file in sync_files:
        print(f'  {file}')

sync_df = None
for filename in ['sync_metadata.csv', 'synchronization.csv', 'sync.csv']:
    test_path = os.path.join(METADATA_PATH, filename)
    if os.path.exists(test_path):
        try:
            sync_df = pd.read_csv(test_path)
            print(f'Loaded sync metadata: {sync_df.shape}')
            display(sync_df.head())
            
            if sync_df is not None:
                offset_columns = [col for col in sync_df.columns if 'offset' in col.lower()]
                for col in offset_columns:
                    print(f'{col}:')
                    print(f'  Mean: {sync_df[col].mean():.4f} seconds')
                    print(f'  Std: {sync_df[col].std():.4f} seconds')
                    
                    plt.figure(figsize=(10, 4))
                    plt.hist(sync_df[col], bins=50, color='skyblue', edgecolor='black', alpha=0.7)
                    plt.title(f'{col} Distribution')
                    plt.xlabel('Offset (seconds)')
                    plt.ylabel('Frequency')
                    plt.grid(True, alpha=0.3)
                    plt.show()
        except Exception as e:
            print(f'Error loading {filename}: {e}')

print('PPG Sync Directory:')
if os.path.exists(PPG_SYNC_PATH):
    ppg_sync_files = []
    for root, dirs, files in os.walk(PPG_SYNC_PATH):
        for file in files[:10]:
            ppg_sync_files.append(os.path.join(root, file))
    
    print(f'Found {len(ppg_sync_files)} PPG sync files')
    for file in ppg_sync_files[:5]:
        size = os.path.getsize(file) / 1024 / 1024
        print(f'  {file} ({size:.2f} MB)')

## 10. Multi-Camera Analysis

In [ ]:
if os.path.exists(VIDEOS_PATH):
    camera_data = {'camera_1': [], 'camera_2': [], 'camera_3': []}
    
    for root, dirs, files in os.walk(VIDEOS_PATH):
        for file in files:
            if 'camera_1' in file:
                camera_data['camera_1'].append(os.path.join(root, file))
            elif 'camera_2' in file:
                camera_data['camera_2'].append(os.path.join(root, file))
            elif 'camera_3' in file:
                camera_data['camera_3'].append(os.path.join(root, file))
    
    print('Camera Distribution:')
    for camera, files in camera_data.items():
        print(f'  {camera}: {len(files)} videos')
    
    if 'file' in df_db.columns:
        df_db['camera'] = df_db['file'].apply(lambda x: 'camera_1' if 'camera_1' in x else ('camera_2' if 'camera_2' in x else ('camera_3' if 'camera_3' in x else 'unknown')))
        
        camera_stats = df_db.groupby('camera').agg({
            'video_duration': ['mean', 'std', 'min', 'max'],
            'video_mean': ['mean', 'std'],
            'ppg_mean': ['mean', 'std']
        }).round(2)
        display(camera_stats)
        
        plt.figure(figsize=(15, 10))
        plt.subplot(2, 2, 1)
        sns.boxplot(x='camera', y='video_duration', data=df_db, palette='Set2')
        plt.title('Video Duration by Camera')
        
        plt.subplot(2, 2, 2)
        sns.boxplot(x='camera', y='video_mean', data=df_db, palette='Set2')
        plt.title('Video Mean Intensity by Camera')
        
        plt.subplot(2, 2, 3)
        sns.boxplot(x='camera', y='ppg_mean', data=df_db, palette='Set2')
        plt.title('PPG Mean by Camera')
        
        plt.subplot(2, 2, 4)
        sns.boxplot(x='camera', y='face_square', data=df_db, palette='Set2')
        plt.title('Face Square by Camera')
        
        plt.tight_layout()
        plt.show()

## 11. Condition Analysis (Rest vs Exercise)

In [ ]:
if os.path.exists(DB_PATH) and 'condition' in df_db.columns:
    condition_stats = df_db.groupby('condition').agg({
        'video_duration': ['mean', 'std', 'min', 'max'],
        'video_mean': ['mean', 'std'],
        'ppg_mean': ['mean', 'std'],
        'face_square': ['mean', 'std']
    }).round(2)
    display(condition_stats)
    
    metrics = ['video_duration', 'video_mean', 'video_std', 'ppg_mean', 'ppg_std', 'face_square']
    
    plt.figure(figsize=(15, 12))
    for i, metric in enumerate(metrics):
        plt.subplot(2, 3, i + 1)
        if metric in df_db.columns:
            sns.boxplot(x='condition', y=metric, data=df_db, palette='Set2')
            plt.title(f'{metric.replace("_", " ").title()} by Condition')
    plt.tight_layout()
    plt.show()
    
    print('Statistical Comparison:')
    for metric in metrics:
        if metric in df_db.columns:
            rest_data = df_db[df_db['condition'] == 'before'][metric].dropna()
            exercise_data = df_db[df_db['condition'] == 'after'][metric].dropna()
            
            if len(rest_data) > 0 and len(exercise_data) > 0:
                mean_diff = exercise_data.mean() - rest_data.mean()
                print(f'  {metric}: Rest={rest_data.mean():.2f}, Exercise={exercise_data.mean():.2f}, Diff={mean_diff:.2f}')

## 12. Data Quality Assessment

In [ ]:
if os.path.exists(DB_PATH):
    numeric_cols = ['video_mean', 'video_std', 'ppg_mean', 'ppg_std', 'video_duration', 'face_square']
    
    plt.figure(figsize=(15, 10))
    for i, col in enumerate(numeric_cols):
        plt.subplot(2, 3, i + 1)
        sns.boxplot(x=df_db[col], color='lightblue')
        plt.title(f'{col.replace("_", " ").title()} - Outlier Detection')
        
        Q1 = df_db[col].quantile(0.25)
        Q3 = df_db[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = ((df_db[col] < lower_bound) | (df_db[col] > upper_bound)).sum()
        outlier_pct = (outliers / len(df_db[col])) * 100
        
        plt.text(0.05, 0.95, f'Outliers: {outlier_pct:.1f}%', 
                transform=plt.gca().transAxes, 
                bbox=dict(facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    plt.show()
    
    df_db['ppg_quality_score'] = (df_db['ppg_std'] / df_db['ppg_std'].max()) * 100
    df_db['video_quality_score'] = (df_db['video_std'] / df_db['video_std'].max()) * 100
    
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.hist(df_db['ppg_quality_score'], bins=50, color='red', edgecolor='black', alpha=0.7)
    plt.title('PPG Quality Score')
    plt.axvline(df_db['ppg_quality_score'].mean(), color='black', linestyle='--')
    
    plt.subplot(1, 2, 2)
    plt.hist(df_db['video_quality_score'], bins=50, color='blue', edgecolor='black', alpha=0.7)
    plt.title('Video Quality Score')
    plt.axvline(df_db['video_quality_score'].mean(), color='black', linestyle='--')
    
    plt.tight_layout()
    plt.show()

## 13. Sample Video and Signal Visualization

In [ ]:
def load_sample_data(video_path, max_frames=10):
    if os.path.exists(video_path):
        try:
            videodata = skvideo.io.vread(video_path)
            print(f'Video loaded: {videodata.shape}')
            
            plt.figure(figsize=(10, 6))
            plt.imshow(videodata[0])
            plt.title(f'First Frame of {os.path.basename(video_path)}')
            plt.axis('off')
            plt.show()
            
            n_frames = min(max_frames, videodata.shape[0])
            fig, axes = plt.subplots(1, n_frames, figsize=(20, 4))
            if n_frames == 1:
                axes = [axes]
            
            for i in range(n_frames):
                axes[i].imshow(videodata[i])
                axes[i].set_title(f'Frame {i}')
                axes[i].axis('off')
            
            plt.suptitle(f'First {n_frames} Frames')
            plt.tight_layout()
            plt.show()
            
            return videodata
        except Exception as e:
            print(f'Error loading video: {e}')
            return None
    return None

sample_video = None
sample_ppg = None
sample_ecg = None

if os.path.exists(VIDEOS_PATH):
    for root, dirs, files in os.walk(VIDEOS_PATH):
        if files:
            sample_video = os.path.join(root, files[0])
            subject_dir = os.path.basename(root)
            
            ppg_subject_dir = os.path.join(PPG_PATH, subject_dir)
            if os.path.exists(ppg_subject_dir):
                ppg_files = os.listdir(ppg_subject_dir)
                if ppg_files:
                    sample_ppg = os.path.join(ppg_subject_dir, ppg_files[0])
            
            ecg_subject_dir = os.path.join(ECG_PATH, subject_dir)
            if os.path.exists(ecg_subject_dir):
                ecg_files = os.listdir(ecg_subject_dir)
                if ecg_files:
                    sample_ecg = os.path.join(ecg_subject_dir, ecg_files[0])
            break

if sample_video:
    print(f'Sample video: {sample_video}')
    videodata = load_sample_data(sample_video)
    
    if sample_ppg and os.path.exists(sample_ppg):
        try:
            ppg_signal = np.load(sample_ppg)
            print(f'PPG signal: {ppg_signal.shape}, Mean={ppg_signal.mean():.2f}')
            
            plt.figure(figsize=(15, 4))
            plt.plot(ppg_signal[:1000], color='red', linewidth=1)
            plt.title('PPG Signal (First 1000 samples)')
            plt.xlabel('Sample Index')
            plt.ylabel('PPG Value')
            plt.grid(True, alpha=0.3)
            plt.show()
        except Exception as e:
            print(f'Error loading PPG: {e}')
    
    if sample_ecg and os.path.exists(sample_ecg):
        try:
            ecg_signal = np.load(sample_ecg)
            print(f'ECG signal: {ecg_signal.shape}, Mean={ecg_signal.mean():.2f}')
            
            plt.figure(figsize=(15, 4))
            plt.plot(ecg_signal[:1000], color='green', linewidth=1)
            plt.title('ECG Signal (First 1000 samples)')
            plt.xlabel('Sample Index')
            plt.ylabel('ECG Value')
            plt.grid(True, alpha=0.3)
            plt.show()
        except Exception as e:
            print(f'Error loading ECG: {e}')

## 14. Summary Statistics and Insights

In [ ]:
if os.path.exists(DB_PATH):
    print('KEY INSIGHTS:')
    print(f'Total videos: {len(df_db)}')
    print(f'Unique subjects: {df_db["subject_id"].nunique() if "subject_id" in df_db.columns else "N/A"}')
    print(f'Average video duration: {df_db["video_duration"].mean():.1f} frames')
    print(f'Average PPG mean: {df_db["ppg_mean"].mean():.2f}')
    print(f'Average PPG std: {df_db["ppg_std"].mean():.2f}')
    print(f'Average face square: {df_db["face_square"].mean():.1f}')
    
    quality_metrics = {
        'Video Quality': df_db['video_quality_score'].mean() if 'video_quality_score' in df_db.columns else 0,
        'PPG Quality': df_db['ppg_quality_score'].mean() if 'ppg_quality_score' in df_db.columns else 0,
        'Data Completeness': (1 - df_db.isnull().mean().mean()) * 100
    }
    
    print('OVERALL DATA QUALITY:')
    for metric, score in quality_metrics.items():
        print(f'  {metric}: {score:.1f}%')
    
    summary_data = {
        'Metric': ['Total Videos', 'Total Subjects', 'Avg Video Duration', 'Avg PPG Mean', 'Avg PPG Std', 'Avg Face Square'],
        'Value': [
            len(df_db),
            df_db['subject_id'].nunique() if 'subject_id' in df_db.columns else 'N/A',
            f"{df_db['video_duration'].mean():.1f} frames",
            f"{df_db['ppg_mean'].mean():.2f}",
            f"{df_db['ppg_std'].mean():.2f}",
            f"{df_db['face_square'].mean():.1f}"
        ]
    }
    
    summary_df = pd.DataFrame(summary_data)
    display(summary_df)

## 15. Appendix: ROI Definitions

### Regions of Interest (ROI) for rPPG:

- **full_face**: 468 landmarks
- **forehead**: [103, 104, 105, 332, 333, 334, 6, 7, 8, 9, 10]
- **left_eye**: landmarks 22-52
- **right_eye**: landmarks 252-282
- **nose**: landmarks 1-20, 195-220
- **mouth**: landmarks 60-80, 290-320
- **left_cheek**: landmarks 0-100, 200-300
- **right_cheek**: landmarks 100-200, 300-400
- **chin**: landmarks 150-170, 370-390
- **left_iris**: landmarks 468-472
- **right_iris**: landmarks 473-477

### Preprocessing Notes:
- Output path: /home/cristic/preprocessed_data
- Faces path: /home/cristic/preprocessed_data/faces/
- Landmarks path: /home/cristic/preprocessed_data/landmarks/
- Use rppglib.face_utils.process_video(video) for processing
- Video loading: skvideo.io.vread(path)